# STEP 1: Setup + Data Load
Install Libraries

In [ ]:
pip install pandas numpy scikit-learn xgboost openpyxl scipy

Load Data

In [ ]:
import pandas as pd

# Load data
train_df = pd.read_excel("/content/Sample_arvyax_reflective_dataset.xlsx")
test_df = pd.read_excel("/content/arvyax_test_inputs_120.xlsx")

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

train_df.head()

Train Shape: (1200, 13)
Test Shape: (120, 11)


,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
0,1,The ocean ambience helped me stop drifting and...,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3
1,2,"I tried to relax during the forest ambience, y...",forest,35,6.0,2,4,evening,calm,tired_face,vague,restless,3
2,3,The forest session slowed my thoughts and I fe...,forest,3,NaN,2,1,night,overwhelmed,happy_face,clear,calm,3
3,4,"the mountain ambience was pleasant, though i c...",mountain,25,7.0,4,4,night,focused,calm_face,vague,neutral,1
4,5,"The rain session gave me a pause, but the pres...",rain,25,5.0,3,5,afternoon,NaN,tense_face,clear,overwhelmed,5


# STEP 2: Data Preprocessing

In [3]:
# Fill missing text
train_df['journal_text'] = train_df['journal_text'].fillna("")
test_df['journal_text'] = test_df['journal_text'].fillna("")

# Fill numeric missing
num_cols = ['sleep_hours', 'energy_level', 'stress_level', 'duration_min']
for col in num_cols:
    train_df[col] = train_df[col].fillna(train_df[col].median())
    test_df[col] = test_df[col].fillna(test_df[col].median())

# Fill categorical
cat_cols = ['ambience_type', 'time_of_day', 'previous_day_mood', 'face_emotion_hint', 'reflection_quality']
for col in cat_cols:
    train_df[col] = train_df[col].fillna("unknown")
    test_df[col] = test_df[col].fillna("unknown")

## STEP 3: Feature Engineering
 Text → TF-IDF

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)

X_text_train = tfidf.fit_transform(train_df['journal_text'])
X_text_test = tfidf.transform(test_df['journal_text'])

In [5]:
# Metadata Encoding
train_meta = train_df.drop(columns=['journal_text', 'emotional_state', 'intensity', 'id'])
test_meta = test_df.drop(columns=['journal_text', 'id'])

# One-hot encoding
train_meta = pd.get_dummies(train_meta)
test_meta = pd.get_dummies(test_meta)

# Align columns
train_meta, test_meta = train_meta.align(test_meta, join='left', axis=1, fill_value=0)

In [7]:
# Combine Features
from scipy.sparse import hstack, csr_matrix
import numpy as np

# Ensure train_meta and test_meta are numerical before converting to sparse
# Convert DataFrames to NumPy arrays with float type, then to sparse matrices
X_train_meta_sparse = csr_matrix(train_meta.astype(float).values)
X_test_meta_sparse = csr_matrix(test_meta.astype(float).values)

X_train = hstack([X_text_train, X_train_meta_sparse])
X_test = hstack([X_text_test, X_test_meta_sparse])

## STEP 4: Train Models

In [9]:
# Emotional State (Classification)
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Encode target labels
le = LabelEncoder()
y_state_encoded = le.fit_transform(train_df['emotional_state'])

state_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False, # Set to False as labels are already encoded
    eval_metric='mlogloss'
)

state_model.fit(X_train, y_state_encoded)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:43:31] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [10]:
# Intensity (Regression)
from xgboost import XGBRegressor

y_intensity = train_df['intensity']

intensity_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1
)

intensity_model.fit(X_train, y_intensity)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

## STEP 5: Predictions

In [11]:
# State prediction
state_pred = state_model.predict(X_test)

# Probabilities (for confidence)
state_prob = state_model.predict_proba(X_test)

# Intensity prediction
intensity_pred = intensity_model.predict(X_test)

# Round intensity (1–5)
intensity_pred = intensity_pred.clip(1,5).round()

##STEP 6: Confidence + Uncertainty

In [12]:
import numpy as np

confidence = np.max(state_prob, axis=1)

uncertain_flag = (confidence < 0.6).astype(int)

## STEP 7: Decision Engine (CORE)

In [13]:
def decide_action(state, intensity, stress, energy, time):

    if stress >= 4:
        return "box_breathing", "now"

    if energy <= 2:
        return "rest", "now"

    if state == "anxious":
        return "grounding", "within_15_min"

    if state == "calm" and energy > 3:
        return "deep_work", "now"

    if time in ["night", "evening"]:
        return "light_planning", "tonight"

    return "journaling", "later_today"

In [14]:
actions = []
timings = []

for i in range(len(test_df)):
    state = state_pred[i]
    intensity = intensity_pred[i]
    stress = test_df.iloc[i]['stress_level']
    energy = test_df.iloc[i]['energy_level']
    time = test_df.iloc[i]['time_of_day']

    action, timing = decide_action(state, intensity, stress, energy, time)

    actions.append(action)
    timings.append(timing)

## STEP 8: Create Final Output

In [15]:
output = pd.DataFrame({
    'id': test_df['id'],
    'predicted_state': state_pred,
    'predicted_intensity': intensity_pred,
    'confidence': confidence,
    'uncertain_flag': uncertain_flag,
    'what_to_do': actions,
    'when_to_do': timings
})

output.to_csv("predictions.csv", index=False)

print(" predictions.csv saved!")

 predictions.csv saved!
